# Day 5 · Mix vs. Rate Decomposition & Information Value

*Work log: a cleaned version of my working notebook. The polished analysis lives in [`../01`–`05`](../).*

**Goals:** test whether the 2016 approval-rate drop came from weaker applicants (mix) or stricter approvals within the same score bands (rate), and compute my first Weight of Evidence / Information Value.

**Key results:** the approval rate fell **5.85 pp** (12.88% → 7.03%). The **mix effect explains 2%**, the **rate effect 98%**: applicants did not get meaningfully worse. Income has **IV = 0.06 (weak)** with a monotonic WOE.

## 1. Setup

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = (Path.cwd().parents[1] / "Data").resolve()   # project_root/Data
con = duckdb.connect()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
# Rebuild the analysis views (DuckDB views are in-memory and reset every session)

# loans: every accepted loan
con.sql(f"""
CREATE OR REPLACE VIEW loans AS
SELECT *
FROM '{DATA}/accepted.parquet'
""")

# loans_clean: drop the 33 file-footer rows and parse the text issue date ('Dec-2015') into a real date
con.sql("""
CREATE OR REPLACE VIEW loans_clean AS
SELECT *,
       strptime(issue_d, '%b-%Y') AS issue_date
FROM loans
WHERE loan_status IS NOT NULL
""")

# cohort: 36-month loans issued 2014-2015, all matured by Dec-2018 so outcomes are final
con.sql("""
CREATE OR REPLACE VIEW cohort AS
SELECT *
FROM loans_clean
WHERE issue_date BETWEEN '2014-01-01' AND '2015-12-31'
  AND TRIM(term) = '36 months'
""")

# rejected: declined applications
con.sql(f"""
CREATE OR REPLACE VIEW rejected AS
SELECT *
FROM '{DATA}/rejected.parquet'
""")

con.sql("SHOW TABLES").df()

,name
0,cohort
1,loans
2,loans_clean
3,rejected


## 2. Credit score fields
Declined applications carry `Risk_Score`; approved loans carry `fico_range_low`. Missing rate = 1 − `COUNT(col)` / `COUNT(*)`.

In [3]:
con.sql("""
SELECT YEAR("Application Date")                              AS app_year,
       COUNT(*)                                              AS n_apps,
       COUNT(Risk_Score)                                     AS n_with_score,
       ROUND(1 - COUNT(Risk_Score) * 1.0 / COUNT(*), 4)      AS pct_missing,
       quantile_cont(Risk_Score, 0.5)                        AS median_score,
       MIN(Risk_Score)                                       AS min_score,
       MAX(Risk_Score)                                       AS max_score
FROM rejected
WHERE YEAR("Application Date") IN (2015, 2016)
GROUP BY app_year
ORDER BY app_year
""").df()

,app_year,n_apps,n_with_score,pct_missing,median_score,min_score,max_score
0,2015,2859379,510451,0.8215,641.0,300.0,990.0
1,2016,4769874,1018297,0.7865,636.0,300.0,990.0


In [4]:
con.sql("""
SELECT YEAR(issue_date)                     AS issue_year,
       COUNT(*)                             AS n_loans,
       MIN(fico_range_low)                  AS min_fico,
       quantile_cont(fico_range_low, 0.5)   AS median_fico,
       MAX(fico_range_low)                  AS max_fico
FROM loans_clean
WHERE YEAR(issue_date) IN (2015, 2016)
GROUP BY issue_year
ORDER BY issue_year
""").df()

,issue_year,n_loans,min_fico,median_fico,max_fico
0,2015,421095,660.0,685.0,845.0
1,2016,434407,660.0,690.0,845.0


* ~80% of declined applications have **no score**: they were rejected **before** a credit pull (basic eligibility checks), so missing ≠ low score.
* The lowest approved FICO is exactly **660** in both years: a hard **knockout rule**.
* Declined applicants' median score (636–641) sits below the cut-off; approved loans' median is 685–690.

## 3. One table of all applications (`UNION ALL`)
Stack approved and declined applications. The source table already tells us the outcome, so `approved` is a constant, not a condition.

In [5]:
con.sql("""
CREATE OR REPLACE VIEW applications AS
SELECT DATE_TRUNC('month', issue_date) AS app_month,
       fico_range_low                  AS score,
       1                               AS approved
FROM loans_clean

UNION ALL      -- not UNION: identical-looking rows are still different applications

SELECT DATE_TRUNC('month', "Application Date"),
       Risk_Score,
       0
FROM rejected
""")

# Sanity check: 2,260,668 approved + 27,648,741 declined
con.sql("""
SELECT approved, COUNT(*) AS n_apps
FROM applications
GROUP BY approved
ORDER BY approved
""").df()

,approved,n_apps
0,0,27648741
1,1,2260668


## 4. Share and approval rate by score band
`PARTITION BY period` makes each period's shares sum to 100%. My first version partitioned by `score_band`, which split each band between the two periods instead; the 100% check caught it.

In [6]:
bands = con.sql("""
WITH labeled AS (
    SELECT CASE WHEN app_month BETWEEN '2015-07-01' AND '2015-12-01' THEN '2015 H2'
                WHEN app_month BETWEEN '2016-07-01' AND '2016-12-01' THEN '2016 H2'
           END AS period,
           CASE WHEN score IS NULL THEN '0. Missing'
                WHEN score < 600   THEN '1. <600'
                WHEN score < 660   THEN '2. 600-659'
                WHEN score < 700   THEN '3. 660-699'
                WHEN score < 750   THEN '4. 700-749'
                ELSE                    '5. 750+'
           END AS score_band,
           approved
    FROM applications
)
SELECT period,
       score_band,
       COUNT(*)                                                           AS n_apps,
       ROUND(COUNT(*) * 1.0 / SUM(COUNT(*)) OVER (PARTITION BY period), 4) AS share,
       ROUND(AVG(approved), 4)                                            AS approval_rate   -- mean of a 0/1 flag
FROM labeled
WHERE period IS NOT NULL
GROUP BY period, score_band
ORDER BY score_band, period
""").df()

print(bands.groupby("period")["share"].sum())   # each period should sum to 1
bands

period
2015 H2    1.0
2016 H2    1.0
Name: share, dtype: float64


,period,score_band,n_apps,share,approval_rate
0,2015 H2,0. Missing,1519062,0.8119,0.0000
1,2016 H2,0. Missing,1951817,0.6778,0.0000
2,2015 H2,1. <600,21424,0.0115,0.0000
3,2016 H2,1. <600,217623,0.0756,0.0000
4,2015 H2,2. 600-659,47039,0.0251,0.0000
5,2016 H2,2. 600-659,278176,0.0966,0.0000
6,2015 H2,3. 660-699,179586,0.0960,0.8441
7,2016 H2,3. 660-699,268169,0.0931,0.4736
8,2015 H2,4. 700-749,85167,0.0455,0.8573
9,2016 H2,4. 700-749,124827,0.0433,0.4922


* Below 660: **0% approval** in both periods.
* The below-660 share rose from 3.7% to 17.2%, but mostly at the expense of *unscored* applicants (81% → 68%), who are also never approved. The share **above 660 stayed ~15%**.
* Above 660, approval **nearly halved** in every band, and fell most for **750+ (87% → 36%)**.

## 5. Mix vs. rate decomposition
`approval rate = Σ share × band rate`. The counterfactual keeps 2015's approval standards but uses 2016's applicant mix.

In [7]:
wide = bands.pivot(index="score_band", columns="period", values=["share", "approval_rate"])

s15 = wide["share"]["2015 H2"]
s16 = wide["share"]["2016 H2"]
r15 = wide["approval_rate"]["2015 H2"]
r16 = wide["approval_rate"]["2016 H2"]

actual_15      = (s15 * r15).sum()
counterfactual = (s16 * r15).sum()
actual_16      = (s16 * r16).sum()

mix_effect   = counterfactual - actual_15
rate_effect  = actual_16 - counterfactual
total_change = actual_16 - actual_15

# Rates are shown in %, differences between rates in percentage points (pp)
print(f"2015 actual:    {actual_15:.2%}")
print(f"2016 actual:    {actual_16:.2%}")
print(f"counterfactual: {counterfactual:.2%}")
print(f"total change:   {total_change * 100:.2f} pp")
print(f"mix effect:     {mix_effect * 100:.2f} pp  ({mix_effect / total_change:.0%} of change)")
print(f"rate effect:    {rate_effect * 100:.2f} pp  ({rate_effect / total_change:.0%} of change)")

2015 actual:    12.88%
2016 actual:    7.03%
counterfactual: 12.76%
total change:   -5.84 pp
mix effect:     -0.12 pp  (2% of change)
rate effect:    -5.72 pp  (98% of change)


**Findings:** the "applicants got worse" hypothesis is **rejected**: 98% of the drop is the rate effect. The remaining question is **policy tightening vs. a funding constraint**. The steepest drop among 750+ applicants points toward funding, since a credit-policy change normally cuts marginal borrowers first.

**Simpson's paradox** is the reason to check mix first: every segment's rate can rise while the overall rate falls, if the mix shifts toward low-rate segments.

## 6. Weight of Evidence and Information Value for income

In [8]:
import numpy as np

iv = con.sql("""
SELECT CASE WHEN annual_inc IS NULL  THEN '0. Missing'
            WHEN annual_inc < 40000  THEN '1. <40k'
            WHEN annual_inc < 60000  THEN '2. 40-60k'
            WHEN annual_inc < 80000  THEN '3. 60-80k'
            WHEN annual_inc < 100000 THEN '4. 80-100k'
            ELSE                          '5. 100k+'
       END AS income_band,
       SUM(CASE WHEN loan_status = 'Fully Paid' THEN 1 ELSE 0 END)                         AS n_good,
       SUM(CASE WHEN loan_status IN ('Charged Off', 'Late (31-120 days)') THEN 1 ELSE 0 END) AS n_bad
FROM cohort
GROUP BY income_band
ORDER BY income_band
""").df()

# Distributions across bins (column totals), not within-bin rates
iv["dist_good"] = iv["n_good"] / iv["n_good"].sum()
iv["dist_bad"]  = iv["n_bad"] / iv["n_bad"].sum()
iv["woe"]       = np.log(iv["dist_good"] / iv["dist_bad"])   # < 0: bads over-represented (riskier)
iv["iv_bin"]    = (iv["dist_good"] - iv["dist_bad"]) * iv["woe"]

print(iv["dist_good"].sum(), iv["dist_bad"].sum())    # both should be 1 (up to float rounding)
print("Income IV:", round(iv["iv_bin"].sum(), 4))
iv

0.9999999999999999 1.0
Income IV: 0.0615


,income_band,n_good,n_bad,dist_good,dist_bad,woe,iv_bin
0,1. <40k,66795.0,16293.0,0.175246,0.252573,-0.365508,0.028264
1,2. 40-60k,102085.0,18887.0,0.267835,0.292785,-0.089069,0.002222
2,3. 60-80k,83484.0,13387.0,0.219032,0.207525,0.053970,0.000621
3,4. 80-100k,51866.0,7077.0,0.136078,0.109707,0.215412,0.005681
4,5. 100k+,76919.0,8864.0,0.201808,0.137409,0.384354,0.024752


**Findings**
* **IV = 0.0615: weak** (0.02–0.1). Income separates risk, but only modestly; it needs to be combined with other variables.
* **WOE is monotonic**: −0.37 (< \$40k) → +0.38 (≥ \$100k). Negative WOE means bads are over-represented in that bin, not that the bin has no goods.
* A bin with no bads would give an infinite WOE, one more reason tiny bins are merged.
* **Next:** IV for every application-time variable to shortlist features for the PD model.